In [1]:
import numpy as np
import random
import typing
from pathlib import Path
import random


In [2]:
from sample_people import sample_people
N = 5
people = sample_people(N=N, seed=42)
for p in people:
    print(p)
print()

{'id': 0, 'first_name': 'Chase', 'middle_name': 'Jace', 'last_name': 'May', 'birthday': 24, 'birthmonth': 'February', 'birthyear': 1873, 'birthcity': 'Dayton, OH', 'university': 'Baruch College', 'field': 'Management', 'company1name': 'Disney', 'company1city': 'Burbank, CA'}
{'id': 2, 'first_name': 'James', 'middle_name': 'Jesus', 'last_name': 'Clay', 'birthday': 2, 'birthmonth': 'January', 'birthyear': 1723, 'birthcity': 'Honolulu, HI', 'university': 'Thomas Jefferson University', 'field': 'Data Analytics', 'company1name': 'State Farm Insurance', 'company1city': 'Bloomington, IL'}
{'id': 4, 'first_name': 'Oliver', 'middle_name': 'Dylan', 'last_name': 'Moody', 'birthday': 18, 'birthmonth': 'April', 'birthyear': 1883, 'birthcity': 'Hollywood, FL', 'university': 'Baruch College', 'field': 'Information Systems Management', 'company1name': 'Allstate', 'company1city': 'Northbrook, IL'}
{'id': 6, 'first_name': 'Peyton', 'middle_name': 'Jonah', 'last_name': 'Bush', 'birthday': 15, 'birthmonth

In [3]:
from bio_text import get_text_simple3, bio_stream

p = people[0]

# Determinism: same exposure → same string
assert get_text_simple3(p, exposure=0) == get_text_simple3(p, exposure=0)

# Variation: different exposure → different paraphrase
assert get_text_simple3(p, exposure=0) != get_text_simple3(p, exposure=1)

# Inspect a few
for e in range(3):
    print(f"--- exposure {e} ---")
    print(get_text_simple3(p, exposure=e))
    print()

# Stream test: first 5 bios from a small dataset
small = sample_people(N=10, seed=0)
for i, (pid, ex, text) in enumerate(bio_stream(small, K=3, master_seed=0)):
    print(f"[person {pid}, exposure {ex}] {text[:80]}...")
    if i >= 4:
        break


--- exposure 0 ---
 Chase Jace May pays tribute to the day they were born, February 24, 1873. He feels a deep connection to Dayton, OH. He had access to state-of-the-art facilities and laboratories at Baruch College. He pursued a degree in Management. He worked with clients and customers of Disney. He played a role in the business sector of Burbank, CA.

--- exposure 1 ---
 Chase Jace May took their first breath on February 24, 1873. He has a deep sense of nostalgia for Dayton, OH. He was part of a vibrant and diverse student community at Baruch College. He focused on Management during their studies. He contributed to the success of Disney. He played a role in the business sector of Burbank, CA.

--- exposure 2 ---
 Chase Jace May came into this world on February 24, 1873. He first saw the light of day in Dayton, OH. He graduated from Baruch College. He specialized in Management with a focus on practical applications. He worked diligently at Disney to achieve their goals. He joined the

In [4]:
from tokenize_pack import tokenize_and_pack, PackedTokenDataset
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# GPT-2 has no real BOS/EOS distinction — it uses <|endoftext|> for both.
EOT_ID = tokenizer.eos_token_id          # 50256
VOCAB_SIZE = tokenizer.vocab_size        # 50257

K = 50
stream = bio_stream(people, K=K, master_seed=0, shuffle_seed=1)

n_tokens, n_seq = tokenize_and_pack(
    tokenizer,
    stream,
    n_bios_total=N * K,
    out_path="cache/bios_n1k_k100.bin",
    seq_len=512,
)
print(f"Wrote {n_tokens:,} tokens → {n_seq:,} sequences of 512.")

ds = PackedTokenDataset("cache/bios_n1k_k100.bin", seq_len=512)
print(f"Dataset has {len(ds)} sequences.")
sample = ds[0]
print(sample["input_ids"][:30])
print(tokenizer.decode(sample["input_ids"][:60]))


/Users/efmac/Code/Projet Code/CRL-Interp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 250/250 [00:00<00:00, 95638.09it/s]

Wrote 18,216 tokens → 35 sequences of 512.
Dataset has 35 sequences.
tensor([50256, 35228, 45447,  5511,  1718,   511,   717,  8033,   319,  3267,
         1315,    11,  1596,  4869,    13,   679, 20675,   511, 15587,   736,
          284,  5401,  5652,    11,  7257,    13,   679, 25050,   422,   262])
<|endoftext|> Peyton Jonah Bush took their first breath on October 15, 1771. He traces their origins back to Los Angeles, CA. He benefited from the resources and facilities provided by University of California, Riverside. He specialized in Anthropology. He was employed at Hulu, a respected company. He was part of


### Full Setup

In [5]:
from sample_people import sample_people
from bio_text import bio_stream
from tokenize_pack import tokenize_and_pack, PackedTokenDataset, build_vocab_remap, remap_token_file, decode_from_remapped
from transformers import GPT2Tokenizer

N = 1000          # start tiny — verify pipeline before scaling
K = 100
people = sample_people(N=N, seed=0)
stream = bio_stream(people, K=K, master_seed=0, shuffle_seed=1)


tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# GPT-2 has no real BOS/EOS distinction — it uses <|endoftext|> for both.
EOT_ID = tokenizer.eos_token_id          # 50256
VOCAB_SIZE = tokenizer.vocab_size        # 50257

fullTokenPath="cache/bios_n1k_k100_temporary.bin"
reducedTokenPath="cache/bios_n1k_k100_reduced.bin"

n_tokens, n_seq = tokenize_and_pack(
    tokenizer,
    stream,
    n_bios_total=N * K,
    out_path=fullTokenPath,
    seq_len=512,
)
print(f"Wrote {n_tokens:,} tokens → {n_seq:,} sequences of 512.")

old_to_new, new_to_old, n_unique_tokens = build_vocab_remap(fullTokenPath)

remapped_path = remap_token_file(
    fullTokenPath,
    reducedTokenPath,
    old_to_new
)

ds = PackedTokenDataset(reducedTokenPath, seq_len=512)
print(f"Dataset has {len(ds)} sequences.")
sample = ds[0]
print(sample["input_ids"][:30])
print(decode_from_remapped(sample["input_ids"][:60], new_to_old, tokenizer))


100%|██████████| 100000/100000 [00:03<00:00, 28962.52it/s]


Wrote 7,457,956 tokens → 14,566 sequences of 512.
Dataset has 14566 sequences.
tensor([2902, 1802, 1498,   31, 1574,  114,  824,   79,  649,  375,    4,  342,
        1062,    6,  360,  120,  801,  138, 1057,    4, 1055,    6,  360, 2143,
         138,   44,  765,   60, 1102,  575])
<|endoftext|> Emily Brooklynn Arthur was born on January 14, 1881. She hails from Jackson, MS. She benefited from the resources and facilities provided by Indiana University - Purdue University Indianapolis. She applied their knowledge of Statistics to real-world problems. She contributed their skills to the economic development of Milwaukee
